# NB24 스테이블코인 — 24장

『법률가를 위한 블록체인 — 기술과 경제학으로 읽기』의 장별 노트북입니다. 본문의 코드와 같습니다.

- 모든 코드는 원리를 눈으로 보기 위한 **장난감 구현**입니다. 실제 자산·실제 개인키·실제 거래에 쓰지 마십시오(실사용 금지).
- 외부 API를 부르지 않습니다. 네트워크 없이 실행됩니다. 동봉 스냅샷 데이터가 필요한 셀은 그 데이터를 셀 안에 적어 두었습니다.
- 표준 라이브러리만 씁니다(Python 3.13에서 실행·확인). 설치 셀은 없습니다.
- 출력은 엮은이가 실행한 결과를 그대로 담아 두었습니다. 다시 실행하면 같은 값이 나와야 합니다(난수는 시드를 고정했습니다).

In [ ]:
# NB24-1: 다이아몬드–딥비그의 두 균형 — 요구불예금 r1 > 1, 1기 총인출 비율 f 에 따라 「기다림」의 몫 V2 가 「인출」 r1 보다 큰가 (공부용)
R, r1, t = 1.5, 1.2, 0.25          # 2기 수익 R, 1기 약정 지급 r1(> 1), 유형 1(꼭 1기에 써야 하는 사람)의 비율 t

def V2(f):                          # 식 (7): 기다린 사람의 몫
    return max((1 - r1 * f) * R / (1 - f), 0.0) if f < 1 else 0.0

print(f"R = {R}, r1 = {r1}, 유형 1 비율 t = {t}. 은행이 바닥나는 인출 비율 1/r1 = {1 / r1:.3f}")
print(f"{'1기 총인출 f':>12} {'기다린 몫 V2(f)':>16} {'인출 몫 r1':>10} {'유형 2의 최선':>12}")
for f in [t, 0.40, 0.55, 0.70, 0.80, 0.83, 0.90, 1.00]:
    v = V2(f)
    print(f"{f:>12.2f} {v:>16.3f} {r1:>10.2f} {'기다림' if v > r1 else '인출(런)':>12}")
# 기다림이 인출보다 못해지는 경계: (1 − r1 f)R/(1 − f) = r1  →  f* = (R − r1)/(r1 (R − 1))
f_star = (R - r1) / (r1 * (R - 1))
print(f"\n경계 f* = (R − r1)/(r1(R − 1)) = {f_star:.3f}. 남들이 이보다 많이 인출하리라 믿으면 나도 인출하는 것이 최선이다 — 믿음이 결과를 정한다.")
print(f"f = t = {t} (정상)에서 유형 2는 기다리고 {V2(t):.3f} 를 받는다(> r1). f = 1 (런)에서는 모두가 평균 1 을 나눠 갖고 뒷줄은 0 — 은행이 없었을 때보다 모두 나쁘다.")
print("\n태환 정지(f 가 정지점에 닿으면 1기 지급 중단)를 넣으면 기다린 사람의 몫이 r1 아래로 떨어질 수 없어 유형 2 는 기다리는 것이 우월전략이 된다 — t 를 알 때.")

R = 1.5, r1 = 1.2, 유형 1 비율 t = 0.25. 은행이 바닥나는 인출 비율 1/r1 = 0.833
    1기 총인출 f      기다린 몫 V2(f)    인출 몫 r1     유형 2의 최선
        0.25            1.400       1.20          기다림
        0.40            1.300       1.20          기다림
        0.55            1.133       1.20        인출(런)
        0.70            0.800       1.20        인출(런)
        0.80            0.300       1.20        인출(런)
        0.83            0.035       1.20        인출(런)
        0.90            0.000       1.20        인출(런)
        1.00            0.000       1.20        인출(런)

경계 f* = (R − r1)/(r1(R − 1)) = 0.500. 남들이 이보다 많이 인출하리라 믿으면 나도 인출하는 것이 최선이다 — 믿음이 결과를 정한다.
f = t = 0.25 (정상)에서 유형 2는 기다리고 1.400 를 받는다(> r1). f = 1 (런)에서는 모두가 평균 1 을 나눠 갖고 뒷줄은 0 — 은행이 없었을 때보다 모두 나쁘다.

태환 정지(f 가 정지점에 닿으면 1기 지급 중단)를 넣으면 기다린 사람의 몫이 r1 아래로 떨어질 수 없어 유형 2 는 기다리는 것이 우월전략이 된다 — t 를 알 때.


In [ ]:
# NB24-2: 준비자산 구성 × 상환 요구 → 디페그 되먹임 — 유형 A·B·C 는 구조의 예시이지 어느 코인의 장부가 아니다 (공부용)
# 되먹임 규칙(이 책의 단순화): 상환 가능 비율 RR < 1 이거나 발행자가 창구를 닫으면 2차 시장 가격이 RR 쪽으로 내려가고,
# 가격이 1 아래면 추가 보유자가 상환에 나선다(아래 '민감도' 만큼). 가격·수량은 전부 모형 변수다.
def run_dynamics(reserves, q0, sensitivity=1.5, rounds=6, window_open=True):
    """reserves: [(이름, 비중, 처분가능몫 ℓ, 급매 할인 h)], 비중 합 = 1(담보율 1.0). q0: 처음 상환 요구 비율."""
    S = 1.0                                   # 발행량 1 로 정규화
    q, price, history = q0, 1.0, []
    redeemed = 0.0
    for r in range(1, rounds + 1):
        liquid = sum(w * l * (1 - h) for _, w, l, h in reserves)        # 지금 당장 처분 가능한 가치(발행량 1 기준)
        demand = min(q, 1.0) - redeemed
        rr = liquid / demand if demand > 0 else float("inf")
        if window_open and rr >= 1:
            redeemed += demand; price = 1.0
            history.append((r, q, round(rr, 2), price, "액면 상환 완료"))
            break
        price = max(min(rr, 1.0), 0.0) if window_open else max(price - 0.05, 0.0)   # 창구가 닫히면 가격이 미끄러진다(규칙)
        history.append((r, round(q, 3), round(rr, 2), round(price, 3), "창구 열림" if window_open else "창구 닫힘"))
        q = min(1.0, q + sensitivity * (1 - price) * (1 - q))          # 가격이 1 아래면 더 많은 보유자가 상환에 나선다
    return history

types = {
    "A 현금·국채 위주":            [("현금", 0.20, 1.0, 0.0), ("단기국채", 0.75, 1.0, 0.005), ("은행예금", 0.05, 1.0, 0.0)],
    "B 예금·어음 섞임":            [("현금", 0.05, 1.0, 0.0), ("단기국채", 0.40, 1.0, 0.005), ("은행예금", 0.30, 1.0, 0.0), ("기업어음", 0.25, 1.0, 0.05)],
    "B′ B에서 은행예금 전부 동결": [("현금", 0.05, 1.0, 0.0), ("단기국채", 0.40, 1.0, 0.005), ("은행예금(동결)", 0.30, 0.0, 0.0), ("기업어음", 0.25, 1.0, 0.05)],
    "C 암호자산 초과담보(처음 담보율 1.5 → 담보 45% 급락 뒤 0.83)": [("담보 암호자산 — 급락 45% 뒤 잔존 가치, 급매 할인 15%", 1.5 * 0.55, 1.0, 0.15)],
}
for name, res in types.items():
    cr = sum(w for _, w, _, _ in res)
    print(f"\n[{name}] 담보율 {cr:.2f}")
    for q0 in [0.10, 0.40, 0.80]:
        hist = run_dynamics(res, q0)
        last = hist[-1]
        print(f"  처음 요구 {q0:.0%}: {len(hist)}라운드 → 요구 {last[1]:.0%}, 상환가능비율 {last[2]}, 가격 {last[3]} ({last[4]})")

print("\n[B′ 에서 발행자가 창구를 닫은 경우] — 2차 시장이 있으면 창구를 닫아도 런은 가격으로 이어진다")
for row in run_dynamics(types["B′ B에서 은행예금 전부 동결"], 0.40, window_open=False):
    print("  라운드 {} 요구 {} RR {} 가격 {} {}".format(*row))


[A 현금·국채 위주] 담보율 1.00
  처음 요구 10%: 1라운드 → 요구 10%, 상환가능비율 9.96, 가격 1.0 (액면 상환 완료)
  처음 요구 40%: 1라운드 → 요구 40%, 상환가능비율 2.49, 가격 1.0 (액면 상환 완료)
  처음 요구 80%: 1라운드 → 요구 80%, 상환가능비율 1.25, 가격 1.0 (액면 상환 완료)

[B 예금·어음 섞임] 담보율 1.00
  처음 요구 10%: 1라운드 → 요구 10%, 상환가능비율 9.86, 가격 1.0 (액면 상환 완료)
  처음 요구 40%: 1라운드 → 요구 40%, 상환가능비율 2.46, 가격 1.0 (액면 상환 완료)
  처음 요구 80%: 1라운드 → 요구 80%, 상환가능비율 1.23, 가격 1.0 (액면 상환 완료)

[B′ B에서 은행예금 전부 동결] 담보율 1.00
  처음 요구 10%: 1라운드 → 요구 10%, 상환가능비율 6.85, 가격 1.0 (액면 상환 완료)
  처음 요구 40%: 1라운드 → 요구 40%, 상환가능비율 1.71, 가격 1.0 (액면 상환 완료)
  처음 요구 80%: 6라운드 → 요구 97%, 상환가능비율 0.7, 가격 0.704 (창구 열림)

[C 암호자산 초과담보(처음 담보율 1.5 → 담보 45% 급락 뒤 0.83)] 담보율 0.83
  처음 요구 10%: 1라운드 → 요구 10%, 상환가능비율 7.01, 가격 1.0 (액면 상환 완료)
  처음 요구 40%: 1라운드 → 요구 40%, 상환가능비율 1.75, 가격 1.0 (액면 상환 완료)
  처음 요구 80%: 6라운드 → 요구 96%, 상환가능비율 0.73, 가격 0.726 (창구 열림)

[B′ 에서 발행자가 창구를 닫은 경우] — 2차 시장이 있으면 창구를 닫아도 런은 가격으로 이어진다
  라운드 1 요구 0.4 RR 1.71 가격 0.95 창구 닫힘
  라운드 2 요구 0.445 RR 1.54 가격 0.9 창구 닫힘
  라운드 3 요구 0.528 RR 1.3 가격 0.85

In [ ]:
# NB24-3: 알고리듬형의 반사적 되먹임 — 「S 하나 = 1달러어치 L」 교환 약속, L 의 수요곡선, 뒷받침이 깨지면 커지는 상환 셋만으로 나선이 생기는지 본다 (공부용, 모형 변수)
def algorithmic_spiral(S0, L0, p0, eps=1.0, rounds=10, base_redeem=0.05, panic=0.6, shock_round=3, shock=0.3):
    """L 의 수요곡선: p_L = D / L**(1+eps). eps > 0 이면 공급이 늘 때 가격이 비례 이상으로 내린다(수요가 비탄력적; eps=1 이면 L 시가총액 = D/L).
    매 라운드 S 보유자 일부가 S 를 태우고 1/p_L 개의 L 을 받는다. 뒷받침(L 시가총액/S)이 1 아래면 상환 비율이 panic 만큼 커진다.
    shock_round 에 수요 수준 D 가 shock 배로 준다. 모든 숫자는 모형값이다."""
    S, L = float(S0), float(L0)
    D = p0 * L ** (1 + eps)
    print(f"{'라운드':>4} {'S 발행량':>9} {'L 발행량':>11} {'p_L':>8} {'L 시가총액/S':>12} {'이번 상환 비율':>12}")
    for r in range(1, rounds + 1):
        if r == shock_round:
            D *= shock                                        # 수요 충격(모형 변수)
        p_L = D / L ** (1 + eps)
        backing = L * p_L / S                                 # S 하나당 L 전체로 돌려줄 수 있는 달러어치
        share = base_redeem + (panic * (1 - backing) if backing < 1 else 0.0)
        share = min(share, 0.9)
        print(f"{r:>4} {S:>9,.0f} {L:>11,.0f} {p_L:>8.4f} {backing:>12.2f} {share:>12.2f}")
        burn = S * share
        S -= burn
        L += burn / p_L                                       # 태운 S 만큼 「1달러어치」의 L 을 새로 찍는다
    return S, L, D / L ** (1 + eps)

S_end, L_end, p_end = algorithmic_spiral(S0=1_000, L0=2_000, p0=1.0)
print(f"\n끝: S {S_end:,.0f}, L {L_end:,.0f}, p_L {p_end:.4f}.")
print("수요 충격 뒤 L 의 시가총액이 S 아래로 떨어지면(BIS 의 조건이 깨지면) 상환이 L 을 더 찍고, 늘어난 공급이 p_L 을 더 내리며, 그것이 상환을 더 부른다 — 나선.")
print("이 표의 숫자는 어떤 자료에도 맞추지 않은 모형값이다. 보이는 것은 모양뿐이다: 뒷받침이 「다른 토큰의 수요」일 때 그 수요가 사라지면 뒷받침도 사라진다.")

 라운드     S 발행량       L 발행량      p_L     L 시가총액/S     이번 상환 비율
   1     1,000       2,000   1.0000         2.00         0.05
   2       950       2,050   0.9518         2.05         0.05
   3       902       2,100   0.2721         0.63         0.27
   4       659       2,996   0.1337         0.61         0.29
   5       471       4,400   0.0620         0.58         0.30
   6       328       6,699   0.0267         0.55         0.32
   7       222      10,663   0.0106         0.51         0.35
   8       145      17,966   0.0037         0.46         0.37
   9        91      32,602   0.0011         0.40         0.41
  10        54      65,409   0.0003         0.34         0.45

끝: S 30, L 151,116, p_L 0.0001.
수요 충격 뒤 L 의 시가총액이 S 아래로 떨어지면(BIS 의 조건이 깨지면) 상환이 L 을 더 찍고, 늘어난 공급이 p_L 을 더 내리며, 그것이 상환을 더 부른다 — 나선.
이 표의 숫자는 어떤 자료에도 맞추지 않은 모형값이다. 보이는 것은 모양뿐이다: 뒷받침이 「다른 토큰의 수요」일 때 그 수요가 사라지면 뒷받침도 사라진다.
